In [1]:
tabla ='qtiaa10'
col_tabla = 'solopefec'

In [2]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2

In [3]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [4]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)

now = datetime.now()

# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

# c1= text(query)
# connection2.execute(c1)

In [5]:
fecha = '01/05/23'
# fecha= '2023-05-01'

In [6]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

query0 = f"""
select
d1.INFOPEORICENASICOD,
d1.INFOPECENASICOD,
d1.INFOPESOLOPENUM,
d1.INFOPEINFANE,
d1.INFOPEANESECUE,
d1.INFOPEANEANECOD,
d1.INFOPEANEESTREGCOD,

a1.solopefec as solopefec,  
a1.SOLOPEACTMEDNUM,
a1.SOLOPEBUSPACSECNUM
from sgss.{tabla} d1
left outer join qtsop10 a1 on a1.SOLOPEORICENASICOD = d1.INFOPEORICENASICOD
and a1.SOLOPECENASICOD    = d1.INFOPECENASICOD
and a1.SOLOPENUM    = d1.INFOPESOLOPENUM

where a1.{col_tabla}>='{fecha}'
"""

base2 = pd.read_sql_query(query0,con=connection0)


connection0.close()

In [7]:
base2.shape

(32941, 10)

In [8]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32941 entries, 0 to 32940
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   infopeoricenasicod  32941 non-null  object        
 1   infopecenasicod     32941 non-null  object        
 2   infopesolopenum     32941 non-null  int64         
 3   infopeinfane        32941 non-null  int64         
 4   infopeanesecue      32941 non-null  int64         
 5   infopeaneanecod     32941 non-null  object        
 6   infopeaneestregcod  32941 non-null  object        
 7   solopefec           32941 non-null  datetime64[ns]
 8   solopeactmednum     32941 non-null  int64         
 9   solopebuspacsecnum  32941 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(4)
memory usage: 2.5+ MB


In [9]:
#CREAMOS LA TABLA TEMPORAL
base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

941

In [10]:
base2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32941 entries, 0 to 32940
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   infopeoricenasicod  32941 non-null  object        
 1   infopecenasicod     32941 non-null  object        
 2   infopesolopenum     32941 non-null  int64         
 3   infopeinfane        32941 non-null  int64         
 4   infopeanesecue      32941 non-null  int64         
 5   infopeaneanecod     32941 non-null  object        
 6   infopeaneestregcod  32941 non-null  object        
 7   solopefec           32941 non-null  datetime64[ns]
 8   solopeactmednum     32941 non-null  int64         
 9   solopebuspacsecnum  32941 non-null  int64         
dtypes: datetime64[ns](1), int64(5), object(4)
memory usage: 2.5+ MB


In [11]:
#LA SUBIMOS AL POSTGRES SQL

query_count_before = text(f"SELECT COUNT(*) FROM {tabla}")
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")


Cantidad de filas en la tabla qtiaa10 antes de la inserción: 649259


In [12]:
#Borramos en caso el ETL se ejecute una segunda vez
borrando=text(f"DELETE FROM {tabla} WHERE {col_tabla} >='{fecha}'")
borrado = connection3.execute(borrando)

In [13]:
query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN infopeoricenasicod TYPE character(1),
ALTER COLUMN infopecenasicod TYPE character(3),
ALTER COLUMN infopesolopenum TYPE numeric(10,0),
ALTER COLUMN infopeinfane TYPE numeric(2,0),
ALTER COLUMN infopeanesecue TYPE numeric(2,0),
ALTER COLUMN infopeaneanecod TYPE character(3),
ALTER COLUMN infopeaneestregcod TYPE character(1),
ALTER COLUMN solopefec TYPE date,
ALTER COLUMN solopeactmednum TYPE numeric(10,0),
ALTER COLUMN solopebuspacsecnum TYPE numeric(10,0);

INSERT INTO {tabla} 
(infopeoricenasicod,infopecenasicod,infopesolopenum,infopeinfane,infopeanesecue,infopeaneanecod,infopeaneestregcod,solopefec,solopeactmednum,solopebuspacsecnum)
SELECT 
infopeoricenasicod,infopecenasicod,infopesolopenum,infopeinfane,infopeanesecue,infopeaneanecod,infopeaneestregcod,solopefec,solopeactmednum,solopebuspacsecnum
FROM tmp_{tabla};
"""

c1= text(query)
connection3.execute(c1)

In [14]:
query_count_after = text(f"SELECT COUNT(*) FROM {tabla}")
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla qtiaa10 después de la inserción: 649651
La cantidad de filas insertadas fue de: 392


In [15]:
#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
"""
c2= text(query2)
cursor=connection3.execute(c2)

In [16]:
connection1.close()
connection2.close()
connection3.close()

In [17]:
engine1.dispose()
engine2.dispose()
engine3.dispose()

AYUDA PARA EXTRAER COLUMNAS Y ESTRUCTURA DE TABLAS (NO ES PARTE DEL ETL)

In [18]:
import re

cadena = """
    infopeoricenasicod character(1) COLLATE pg_catalog."default",
    infopecenasicod character(3) COLLATE pg_catalog."default",
    infopesolopenum numeric(10,0),
    infopeinfane numeric(2,0),
    infopeanesecue numeric(2,0),
    infopeaneanecod character(3) COLLATE pg_catalog."default",
    infopeaneestregcod character(1) COLLATE pg_catalog."default",
    solopefec date,
    solopeactmednum numeric(10,0),
    solopebuspacsecnum numeric(10,0)
"""

# Reemplaza los caracteres innecesarios
cadena = cadena.replace(" COLLATE pg_catalog.\"default\",", "")
cadena = cadena.replace(" with time zone", "")

# Divide la cadena en una lista de líneas
lineas = cadena.split("\n")

# Crea la cadena de alteración de columnas
cadena_alter = ""
for i, linea in enumerate(lineas):
    palabras = linea.split()
    if len(palabras) >= 2:
        columna = palabras[0]
        tipo = palabras[1]
        if i == len(lineas) - 2:
            # Última línea, agrega punto y coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo};\n"
        else:
            # Otras líneas, agrega coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo},\n"

# Utiliza una expresión regular para eliminar las comas duplicadas
cadena_alter = re.sub(r',+$', ',', cadena_alter, flags=re.MULTILINE)

print(cadena_alter)



import re

nombrecitos = re.findall(r'ALTER COLUMN\s+(\S+)', cadena_alter)
insertado_col = ",".join(nombrecitos)

print(insertado_col)

ALTER COLUMN infopeoricenasicod TYPE character(1),
ALTER COLUMN infopecenasicod TYPE character(3),
ALTER COLUMN infopesolopenum TYPE numeric(10,0),
ALTER COLUMN infopeinfane TYPE numeric(2,0),
ALTER COLUMN infopeanesecue TYPE numeric(2,0),
ALTER COLUMN infopeaneanecod TYPE character(3),
ALTER COLUMN infopeaneestregcod TYPE character(1),
ALTER COLUMN solopefec TYPE date,
ALTER COLUMN solopeactmednum TYPE numeric(10,0),
ALTER COLUMN solopebuspacsecnum TYPE numeric(10,0);

infopeoricenasicod,infopecenasicod,infopesolopenum,infopeinfane,infopeanesecue,infopeaneanecod,infopeaneestregcod,solopefec,solopeactmednum,solopebuspacsecnum
